# Reproducible Renewable Forecasting Pipeline (Synthetic Data)

This notebook implements a transparent **baseline vs optimized** forecasting comparison using a **rolling time-series backtest**.

- **Baseline**: Ridge regression
- **Optimized**: Random Forest with a small grid search (time-series CV)

Outputs are saved to `reports/` and `reports/figures/`.


## 1) Imports and paths

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd()
DATA_DIR = ROOT / "data"
REPORTS_DIR = ROOT / "reports"
FIG_DIR = REPORTS_DIR / "figures"

DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH = DATA_DIR / "synthetic_renewable_timeseries.csv"
DATASET_PATH

## 2) Generate (or load) a synthetic renewable dataset

If you have real data, replace this section by loading your dataset and ensuring you have these columns:

- `timestamp` (datetime)
- meteorological drivers like `irradiance`, `wind_speed`, `temperature_c`, `humidity_pct`
- `renewable_power` (target signal to forecast)


In [ ]:
def generate_synthetic_dataset(n_days: int = 180, seed: int = 7) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    idx = pd.date_range("2025-01-01", periods=n_days * 24, freq="H")
    n = len(idx)

    hour = idx.hour.values
    dayofyear = idx.dayofyear.values

    seasonal = 0.65 + 0.35 * np.sin(2 * np.pi * (dayofyear / 365.25))
    diurnal = np.clip(np.sin(np.pi * (hour - 6) / 12), 0, None)
    cloud = np.clip(rng.normal(0.0, 0.18, size=n), -0.6, 0.6)
    irradiance = np.clip(seasonal * diurnal * (1 + cloud), 0, None)

    base_wind = 7 + 2.0 * np.sin(2 * np.pi * (dayofyear / 365.25 + 0.25))
    ar = rng.normal(0, 0.8, size=n)
    for i in range(1, n):
        ar[i] = 0.85 * ar[i - 1] + ar[i]
    wind_speed = np.clip(base_wind + ar + rng.normal(0, 1.0, size=n), 0, None)

    temperature = 12 + 10 * np.sin(2 * np.pi * (dayofyear / 365.25 - 0.1)) + rng.normal(0, 1.5, size=n)
    humidity = np.clip(55 + 20 * np.sin(2 * np.pi * (dayofyear / 365.25 + 0.05)) + rng.normal(0, 6, size=n), 10, 100)

    pv_power = np.clip(irradiance ** 1.15 + rng.normal(0, 0.03, size=n), 0, None)
    wind_norm = np.clip((wind_speed - 3) / (12 - 3), 0, 1)
    wind_power = np.clip(wind_norm ** 3 + rng.normal(0, 0.04, size=n), 0, 1)
    renewable_power = np.clip(0.55 * pv_power + 0.45 * wind_power + rng.normal(0, 0.03, size=n), 0, 1.2)

    df = pd.DataFrame(
        {
            "timestamp": idx,
            "irradiance": irradiance,
            "wind_speed": wind_speed,
            "temperature_c": temperature,
            "humidity_pct": humidity,
            "hour": hour,
            "dayofyear": dayofyear,
            "renewable_power": renewable_power,
        }
    )

    for col in ["irradiance", "wind_speed", "temperature_c", "humidity_pct"]:
        mask = rng.random(n) < 0.01
        df.loc[mask, col] = np.nan

    return df


if DATASET_PATH.exists():
    df = pd.read_csv(DATASET_PATH, parse_dates=["timestamp"])
else:
    df = generate_synthetic_dataset(n_days=180, seed=7)
    df.to_csv(DATASET_PATH, index=False)

df.head()

## 3) Create supervised learning table (forecast horizon + lags)

In [ ]:
def make_supervised(df: pd.DataFrame, horizon_hours: int = 1, n_lags: int = 24) -> pd.DataFrame:
    df = df.sort_values("timestamp").reset_index(drop=True).copy()
    df["target"] = df["renewable_power"].shift(-horizon_hours)
    for lag in range(1, n_lags + 1):
        df[f"lag_{lag}"] = df["renewable_power"].shift(lag)
    return df.dropna().reset_index(drop=True)


sup = make_supervised(df, horizon_hours=1, n_lags=24)
sup.shape, sup.columns[:10]

## 4) Baseline vs optimized model definitions

In [ ]:
numeric_features = [
    "irradiance",
    "wind_speed",
    "temperature_c",
    "humidity_pct",
    "dayofyear",
    *[f"lag_{i}" for i in range(1, 25)],
]
categorical_features = ["hour"]

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
    ]
)

baseline = Pipeline(steps=[("preprocess", preprocess), ("model", Ridge(alpha=1.0, random_state=0))])

optimized_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocess),
        (
            "model",
            RandomForestRegressor(random_state=0, n_jobs=-1),
        ),
    ]
)

param_grid = {
    "model__n_estimators": [200, 500],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_leaf": [1, 3, 5],
}

optimized = GridSearchCV(
    optimized_pipeline,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=TimeSeriesSplit(n_splits=5),
    n_jobs=-1,
    verbose=0,
)

feature_cols = [c for c in sup.columns if c not in {"target", "timestamp", "renewable_power"}]
feature_cols[:8], len(feature_cols)

## 5) Rolling backtest evaluation

In [ ]:
def rolling_backtest(data: pd.DataFrame, test_size: int = 24 * 7, n_splits: int = 6) -> pd.DataFrame:
    n = len(data)
    fold_starts = np.linspace(0, n - test_size, num=n_splits + 1, dtype=int)[1:]
    rows = []

    for i, start in enumerate(fold_starts, start=1):
        train = data.iloc[:start]
        test = data.iloc[start : start + test_size]

        X_train, y_train = train[feature_cols], train["target"]
        X_test, y_test = test[feature_cols], test["target"]

        baseline.fit(X_train, y_train)
        yhat_base = baseline.predict(X_test)

        optimized.fit(X_train, y_train)
        yhat_opt = optimized.predict(X_test)

        rows.append(
            {
                "fold": i,
                "test_start": test["timestamp"].iloc[0],
                "test_end": test["timestamp"].iloc[-1],
                "baseline_mae": mean_absolute_error(y_test, yhat_base),
                "baseline_rmse": float(np.sqrt(mean_squared_error(y_test, yhat_base))),
                "optimized_mae": mean_absolute_error(y_test, yhat_opt),
                "optimized_rmse": float(np.sqrt(mean_squared_error(y_test, yhat_opt))),
                "best_params": str(getattr(optimized, "best_params_", None)),
            }
        )

    return pd.DataFrame(rows)


metrics = rolling_backtest(sup, test_size=24 * 7, n_splits=6)
metrics

## 6) Plot and save the key figure

In [ ]:
sns.set_theme(style="whitegrid")

long_mae = metrics.melt(
    id_vars=["fold", "test_start"],
    value_vars=["baseline_mae", "optimized_mae"],
    var_name="model",
    value_name="mae",
)
long_mae["model"] = long_mae["model"].map(
    {"baseline_mae": "Baseline (Ridge)", "optimized_mae": "Optimized (RF GridSearch)"}
)

plt.figure(figsize=(10, 4.8))
sns.lineplot(data=long_mae, x="test_start", y="mae", hue="model", marker="o")
plt.title("Rolling backtest MAE (lower is better)")
plt.xlabel("Test window start")
plt.ylabel("MAE")
plt.tight_layout()

png_path = FIG_DIR / "rolling_backtest_mae.png"
plt.savefig(png_path, dpi=200)
plt.close()

metrics_path = REPORTS_DIR / "rolling_backtest_metrics.csv"
metrics.to_csv(metrics_path, index=False)

summary = pd.DataFrame(
    {
        "model": ["Baseline (Ridge)", "Optimized (RF GridSearch)"],
        "MAE_mean": [metrics["baseline_mae"].mean(), metrics["optimized_mae"].mean()],
        "RMSE_mean": [metrics["baseline_rmse"].mean(), metrics["optimized_rmse"].mean()],
    }
)
summary_path = FIG_DIR / "metrics_summary.csv"
summary.to_csv(summary_path, index=False)

png_path, metrics_path, summary_path

## 7) Interpretation checklist

- Do optimized results improve **consistently** across folds?
- Are there folds (e.g., stormy periods) where one model fails?
- If you swap in real data, do the conclusions remain stable across seasons?
